In [31]:
"""
MBTA V3 API — Real-time reliability exploration.

Fetches predictions for every bus route, joins each prediction against
its scheduled time (via include=schedule), and computes a delay-based
reliability score per route + system-wide:

    score = 50% on-time rate + 30% normalized avg delay + 20% (1 - disruption rate)

This mirrors the historical scoring logic (delay, on-time window,
disruption handling) so it's directly comparable to the batch pipeline's
metrics — not just a quick proxy.

Usage:
    python explore_reliability.py
"""

import os
import time

import pandas as pd
import requests
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor


API_KEY = os.environ.get("MBTA_API_KEY", "")
BASE_URL = "https://api-v3.mbta.com"
HEADERS = {"x-api-key": API_KEY} if API_KEY else {}

if not API_KEY:
    print("Warning: MBTA_API_KEY not set — you'll hit the lower unauthenticated rate limit.")


def fetch_routes() -> list[str]:
    """All bus route IDs (route_type=3)."""
    response = requests.get(
        f"{BASE_URL}/routes",
        headers=HEADERS,
        params={"filter[type]": 3},
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()["data"]
    return [route["id"] for route in data]


def fetch_predictions_for_route(route_id: str) -> dict:
    """
    Predicted arrival/departure times for buses currently en route,
    joined against the scheduled time via include=schedule so delay
    can actually be computed (not just a disruption count).
    """
    response = requests.get(
        f"{BASE_URL}/predictions",
        headers=HEADERS,
        params={
            "filter[route]": route_id,
            "include": "schedule,stop,route,trip,vehicle",
        },
        timeout=10,
    )
    
    collected_at = datetime.now(timezone.utc).isoformat()
    response.raise_for_status()
    payload = response.json()

    raw_dict = {
        "collected_at": collected_at,
        "data": payload["data"]
    } 
    return raw_dict
    

def fetch_all_routes_prediction(): 
    route_ids = fetch_routes()
    with ThreadPoolExecutor(max_workers=20) as executor:
        results = list(executor.map(fetch_predictions_for_route, route_ids))

    return results


In [34]:
results = fetch_all_routes_prediction()



In [ ]:
import json
json_data = "\n".join(
    json.dumps(result)
    for result in results
)


'{"collected_at": "2026-09-17T22:25:12.484115+00:00", "data": [{"attributes": {"arrival_time": null, "arrival_uncertainty": null, "departure_time": "2026-09-17T18:49:00-04:00", "departure_uncertainty": null, "direction_id": 1, "last_trip": false, "revenue": "REVENUE", "schedule_relationship": null, "status": null, "stop_sequence": 1, "trip_headsign": null, "update_type": null}, "id": "prediction-78592113-17091-1-741", "relationships": {"route": {"data": {"id": "741", "type": "route"}}, "schedule": {"data": {"id": "schedule-78592113-17091-1", "type": "schedule"}}, "stop": {"data": {"id": "17091", "type": "stop"}}, "trip": {"data": {"id": "78592113", "type": "trip"}}, "vehicle": {"data": {"id": "y1320", "type": "vehicle"}}}, "type": "prediction"}, {"attributes": {"arrival_time": null, "arrival_uncertainty": null, "departure_time": "2026-09-17T18:41:00-04:00", "departure_uncertainty": null, "direction_id": 1, "last_trip": false, "revenue": "REVENUE", "schedule_relationship": null, "status

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

start = time.time()
ids = fetch_routes()

with ThreadPoolExecutor(max_workers=40) as executor:
    results = executor.map(fetch_predictions, ids)

total = sum(results)
end = time.time()

print(f"Elapsed: {end - start:.2f} seconds")
print(total)

Elapsed: 3.44 seconds
14603
